# Stoichiometric Analysis of KNSB Composite Sugar Propellant

## Background

KNSB (Potassium Nitrate / Sorbitol) is a castable solid rocket propellant whose oxidiser–fuel
ratio must be carefully chosen to balance performance, safety, and manufacturability.
To determine that ratio rigorously we need a **balanced combustion equation**.

The reaction under study is:

$$
\text{KNO}_{3(s)} + \text{C}_6\text{H}_{14}\text{O}_{6(s)}
\;\rightarrow\;
\text{CO}_{2(g)} + \text{CO}_{(g)} + \text{H}_2\text{O}_{(g)} + \text{H}_{2(g)}
+ \text{N}_{2(g)} + \text{K}_2\text{CO}_{3(g)} + \text{KOH}_{(g)}
$$

We will
1. implement a **Python translation of the Mazza nullspace algorithm** (Mazza & Canuto,
   *Fundamental Chemistry with MATLAB*, Elsevier 2022, Chapter 2), and
2. use it to balance a thermochemically consistent, simplified form of the equation,
3. derive the ideal stoichiometric mass ratio, and
4. justify why a slightly fuel-rich blend is preferred in practice.

*Ferric oxide (Fe₂O₃) acts as a catalyst and does not appear in the equation.*

---
## 1. Python port of the Mazza nullspace method

### 1.1 Theory (Chapter 2, Mazza & Canuto)

Given $n$ substances (reactants and products) containing $m$ elements, label the signed
atom-count matrix $\mathbf{A} \in \mathbb{Z}^{m \times n}$:

$$
A_{jk} = \begin{cases}
+a_{jk} & \text{substance } k \text{ is a reactant} \\
-a_{jk} & \text{substance } k \text{ is a product}
\end{cases}
$$

where $a_{jk}$ is the number of atoms of element $j$ in substance $k$.
Atom conservation then reads

$$
\mathbf{A}\,\mathbf{x} = \mathbf{0},
\qquad \mathbf{x} = [x_1, \ldots, x_n]^\top \in \mathbb{Z}_{>0}^n
$$

The solution lives in the **nullspace** of $\mathbf{A}$.  
A unique (up to scale) positive integer solution exists when $\dim(\ker\mathbf{A}) = 1$,
i.e. the nullspace is one-dimensional. This is the *minimum-balance stoichiometry*.

Mazza's algorithm (adapted from MATLAB to Python/SymPy):
1. Parse each chemical formula as a symbolic expression.
2. Build $\mathbf{A}$ by reading atom counts from the symbolic tree.
3. Compute $\ker\mathbf{A}$ exactly with `Matrix.nullspace()`.
4. Convert the rational nullspace vector to the minimum positive-integer form.

### 1.2 Environment

In [1]:
from sympy import symbols, Symbol, Mul, Pow, Matrix, Rational, lcm as sym_lcm
from math import gcd
from functools import reduce
from fractions import Fraction

### 1.3 Formula parser

Translates the idea behind MATLAB's `symvar` / `children` traversal:
every symbolic expression is either a bare `Symbol`, a `Pow` (element with
exponent), or a `Mul` (product of the previous two). We walk the expression
tree recursively and collect `{element: count}` pairs.

In [ ]:
def _parse_formula(expr, elem_set: set) -> dict:
    """
    Recursively parse a SymPy symbolic expression into an atom-count dict.

    Mirrors the MATLAB `children()` traversal in Mazza (2.4.3)

    Parameters
    ----------
    expr     : SymPy expression  (Symbol, Pow, or Mul)
    elem_set : set of SymPy symbols  — the chemical elements in the reaction

    Returns
    -------
    dict {symbol: int}  atom counts for this formula
    """
    counts: dict = {}

    if isinstance(expr, Symbol):              # e.g.  O, K, H
        if expr in elem_set:
            counts[expr] = 1

    elif isinstance(expr, Pow):              # e.g.  O**3, H**14
        base, exp = expr.args
        if base in elem_set:
            counts[base] = int(exp)

    elif isinstance(expr, Mul):              # e.g.  K*N*O**3, C**6*H**14*O**6
        for factor in expr.args:
            for elem, n in _parse_formula(factor, elem_set).items():
                counts[elem] = counts.get(elem, 0) + n

    return counts

### 1.4 `stoichiometry()` — the core function

Direct Python equivalent of the `stoichiometry.m` function in Mazza §2.5.1.
The main structural difference from the MATLAB version:
- `Matrix.nullspace()` returns exact rational vectors; no need for a separate `rat()` call.
- We use `functools.reduce` + `math.gcd` for the LCM/GCD reduction.

In [3]:
def stoichiometry(
    elements: list,
    substances: list,
    n_reactants: int,
    label: str = "Reaction",
) -> list:
    """
    Minimum-balance stoichiometry via the Mazza nullspace algorithm.

    Port of `stoichiometry.m` (Mazza & Canuto, Chapter 2) from MATLAB to Python/SymPy.

    Parameters
    ----------
    elements    : list of SymPy symbols — chemical elements present (e.g. [K, N, O, C, H])
    substances  : list of SymPy expressions — all substances, reactants first, then products
                  written as symbolic products: K*N*O**3,  C**6*H**14*O**6, etc.
    n_reactants : int — number of reactant substances (prefix of `substances`)
    label       : str — human-readable reaction name for the printed header

    Returns
    -------
    int_coeffs : list[int] — minimum positive integer stoichiometric coefficients,
                 in the same order as `substances` (reactants then products)

    Raises
    ------
    ValueError  — if the nullspace is empty (no solution) or multi-dimensional
                  (underdetermined system; try removing linearly dependent species)
    """
    m = len(elements)          # number of elements
    n = len(substances)        # number of substances
    elem_set = set(elements)

    print(f"\n{'─'*55}")
    print(f"  Minimum-balance stoichiometry  —  {label}")
    print(f"{'─'*55}")
    print(f"  Elements : {m}   Substances : {n}")

    # ------------------------------------------------------------------ #
    # Step 1 – build the signed balance matrix A  (m × n)
    # Mazza eq. (2.16):  A_{jk} = +count for reactants, −count for products
    # ------------------------------------------------------------------ #
    A = Matrix.zeros(m, n)

    for k, formula in enumerate(substances):
        sign = 1 if k < n_reactants else -1
        atom_counts = _parse_formula(formula, elem_set)
        for j, elem in enumerate(elements):
            A[j, k] = sign * atom_counts.get(elem, 0)

    # ------------------------------------------------------------------ #
    # Step 2 – nullspace (exact, rational)
    # MATLAB: Z = null(A, 'r')  →  SymPy: A.nullspace()
    # ------------------------------------------------------------------ #
    null_vecs = A.nullspace()
    dim_null = len(null_vecs)

    if dim_null == 0:
        raise ValueError(
            "Nullspace is empty — the system is inconsistent. "
            "Check that all elements and species are correctly specified."
        )
    if dim_null > 1:
        print(
            f"  ⚠  WARNING: nullspace dimension = {dim_null} > 1 — system is underdetermined.\n"
            f"     The reaction as written has linearly dependent species and cannot\n"
            f"     be given unique stoichiometric coefficients. Simplify the product set."
        )
        return None

    z = null_vecs[0]   # column vector of Rational entries

    # ------------------------------------------------------------------ #
    # Step 3 – convert to minimum positive integers
    # MATLAB: [nZ,dZ]=rat(Z);  intZ=Z*lcm(sym(dZ))
    # ------------------------------------------------------------------ #
    fracs = [Fraction(str(val)) for val in z]          # exact fractions
    denoms = [f.denominator for f in fracs]

    def lcm2(a: int, b: int) -> int:
        return a * b // gcd(a, b)

    denom_lcm = reduce(lcm2, denoms)                   # LCM of denominators
    int_z = [int(f * denom_lcm) for f in fracs]       # scaled to integers

    # Ensure all positive
    if any(v < 0 for v in int_z):
        int_z = [-v for v in int_z]

    # Divide by GCD to reach minimum norm
    g = reduce(gcd, int_z)
    int_z = [v // g for v in int_z]

    # ------------------------------------------------------------------ #
    # Step 4 – print results table
    # ------------------------------------------------------------------ #
    print(f"\n  {'Substance':<22} {'Coeff':>6}  {'Type'}")
    print(f"  {'─'*22} {'─'*6}  {'─'*8}")
    for k, (formula, coeff) in enumerate(zip(substances, int_z)):
        role = "Reactant" if k < n_reactants else "Product"
        print(f"  {str(formula):<22} {coeff:>6}  {role}")

    return int_z

---
## 2. Full KNSB reaction — infeasibility check

The commonly cited full product set is:

$$
a\,\text{KNO}_3 + b\,\text{C}_6\text{H}_{14}\text{O}_6
\;\rightarrow\;
c\,\text{CO}_2 + d\,\text{CO} + e\,\text{H}_2\text{O} + f\,\text{H}_2
+ g\,\text{N}_2 + h\,\text{K}_2\text{CO}_3 + i\,\text{KOH}
$$

We have 5 elements and 9 species, so $\dim(\ker\mathbf{A}) = 9 - \text{rank}(\mathbf{A}) \geq 9 - 5 = 4$.
A unique balanced equation requires the nullspace to be 1-dimensional, which is impossible here.
The `stoichiometry()` call below confirms this.

In [4]:
# ── Element symbols  (same convention as the MATLAB notebook) ──────────────
K, N, O, C, H = symbols('K N O C H')
elements = [K, N, O, C, H]

# ── Substances: reactants then products ───────────────────────────────────
# KNO3           C6H14O6
reactants_full = [K*N*O**3,  C**6*H**14*O**6]

# CO2      CO     H2O      H2      N2      K2CO3          KOH
products_full = [
    C*O**2, C*O, H**2*O, H**2, N**2,
    K**2*C*O**3,
    K*O*H
]

result_full = stoichiometry(
    elements,
    reactants_full + products_full,
    n_reactants=2,
    label="KNSB — full product set (9 species)"
)


───────────────────────────────────────────────────────
  Minimum-balance stoichiometry  —  KNSB — full product set (9 species)
───────────────────────────────────────────────────────
  Elements : 5   Substances : 9
  ⚠  WARNING: nullspace dimension = 4 > 1 — system is underdetermined.
     The reaction as written has linearly dependent species and cannot
     be given unique stoichiometric coefficients. Simplify the product set.


> **Interpretation**: The nullspace dimension equals the number of free parameters in the
> product distribution. With 9 species and only 5 element constraints the system has a
> 4-dimensional nullspace — no unique balanced equation exists. We must fix the product
> slate to obtain a physically meaningful result.

---
## 3. Simplified KNSB reaction — unique balanced equation

Thermochemical analysis (and experimental validation by Nakka 2025) shows that for
near-stoichiometric KNSB at moderate chamber pressure, the dominant products are
$\text{CO}_2$, $\text{H}_2\text{O}$, $\text{N}_2$, and $\text{K}_2\text{CO}_3$.
Dropping the minority species CO, H₂, and KOH reduces the system to 6 substances,
giving a unique (1-D nullspace) solution:

$$
a\,\text{KNO}_3 + b\,\text{C}_6\text{H}_{14}\text{O}_6
\;\rightarrow\;
c\,\text{CO}_2 + d\,\text{H}_2\text{O} + e\,\text{N}_2 + f\,\text{K}_2\text{CO}_3
$$

In [5]:
# ── Simplified product set ─────────────────────────────────────────────────
#        CO2      H2O       N2      K2CO3
products_simplified = [C*O**2, H**2*O, N**2, K**2*C*O**3]

int_coeffs = stoichiometry(
    elements,
    reactants_full + products_simplified,
    n_reactants=2,
    label="KNSB — simplified product set (6 species)"
)

# Unpack results for later use
a_KNO3, b_C6H14O6, c_CO2, d_H2O, e_N2, f_K2CO3 = int_coeffs


───────────────────────────────────────────────────────
  Minimum-balance stoichiometry  —  KNSB — simplified product set (6 species)
───────────────────────────────────────────────────────
  Elements : 5   Substances : 6

  Substance               Coeff  Type
  ────────────────────── ──────  ────────
  K*N*O**3                   26  Reactant
  C**6*H**14*O**6             5  Reactant
  C*O**2                     17  Product
  H**2*O                     35  Product
  N**2                       13  Product
  C*K**2*O**3                13  Product


In [6]:
# Pretty-print the balanced equation
print("\nBalanced combustion equation:")
print(
    f"  {a_KNO3} KNO₃  +  {b_C6H14O6} C₆H₁₄O₆"
    f"  →  {c_CO2} CO₂  +  {d_H2O} H₂O"
    f"  +  {e_N2} N₂  +  {f_K2CO3} K₂CO₃"
)

# Verify atom conservation
print("\nAtom-conservation check:")
atoms_left  = {"K": a_KNO3, "N": a_KNO3,
               "O": 3*a_KNO3 + 6*b_C6H14O6,
               "C": 6*b_C6H14O6, "H": 14*b_C6H14O6}
atoms_right = {"K": 2*f_K2CO3, "N": 2*e_N2,
               "O": 2*c_CO2 + d_H2O + 3*f_K2CO3,
               "C": c_CO2 + f_K2CO3, "H": 2*d_H2O}

for elem in ["K", "N", "O", "C", "H"]:
    l, r = atoms_left[elem], atoms_right[elem]
    status = "✓" if l == r else "✗"
    print(f"  {elem}: left={l:4d}  right={r:4d}  {status}")


Balanced combustion equation:
  26 KNO₃  +  5 C₆H₁₄O₆  →  17 CO₂  +  35 H₂O  +  13 N₂  +  13 K₂CO₃

Atom-conservation check:
  K: left=  26  right=  26  ✓
  N: left=  26  right=  26  ✓
  O: left= 108  right= 108  ✓
  C: left=  30  right=  30  ✓
  H: left=  70  right=  70  ✓


---
## 4. Stoichiometric mass ratio derivation

### 4.1 O₂-demand approach

The O₂ demand of sorbitol (complete combustion) is:

$$
\text{C}_6\text{H}_{14}\text{O}_6 + 6\,\text{O}_2
\;\rightarrow\;
6\,\text{CO}_2 + 7\,\text{H}_2\text{O}
$$

Each mole of KNO₃ supplies 3 oxygen atoms = 1.5 mol O₂-equivalent,
so the stoichiometric KNO₃:sorbitol molar ratio is $6 / 1.5 = 4$.

In [7]:
# ── Molar masses (g/mol) ──────────────────────────────────────────────────
M_KNO3   = 39.102 + 14.007 + 3 * 15.999          # K + N + 3O
M_sorb   = 6*12.011 + 14*1.008 + 6*15.999         # C6H14O6

print(f"Molar mass KNO₃   : {M_KNO3:.5f} g/mol")
print(f"Molar mass sorbitol: {M_sorb:.5f} g/mol")

# ── Molar ratio from balanced equation ───────────────────────────────────
molar_ratio_KNO3_to_sorb = a_KNO3 / b_C6H14O6
print(f"\nMolar ratio KNO₃ : sorbitol  =  {a_KNO3} : {b_C6H14O6}"
      f"  =  {molar_ratio_KNO3_to_sorb:.1f} : 1")

# ── Mass of KNO₃ needed per mol sorbitol ─────────────────────────────────
mass_KNO3_per_mol_sorb = molar_ratio_KNO3_to_sorb * M_KNO3
print(f"Mass of KNO₃ per mol sorbitol: {mass_KNO3_per_mol_sorb:.5f} g")

# ── Ideal mass fractions ──────────────────────────────────────────────────
total_mass = mass_KNO3_per_mol_sorb + M_sorb
wt_KNO3    = mass_KNO3_per_mol_sorb / total_mass
wt_sorb    = M_sorb / total_mass

print(f"\nIdeal stoichiometric mass fractions:")
print(f"  KNO₃    :  {wt_KNO3*100:.2f} %")
print(f"  Sorbitol:  {wt_sorb*100:.2f} %")

# ── Mazza-style O/F ratio (oxidiser / fuel mass ratio) ───────────────────
OF_ratio = mass_KNO3_per_mol_sorb / M_sorb
print(f"\nO/F mass ratio (KNO₃ / sorbitol): {OF_ratio:.4f}")

Molar mass KNO₃   : 101.10600 g/mol
Molar mass sorbitol: 182.17200 g/mol

Molar ratio KNO₃ : sorbitol  =  26 : 5  =  5.2 : 1
Mass of KNO₃ per mol sorbitol: 525.75120 g

Ideal stoichiometric mass fractions:
  KNO₃    :  74.27 %
  Sorbitol:  25.73 %

O/F mass ratio (KNO₃ / sorbitol): 2.8860


### 4.2 Summary

| Quantity | Value |
|---|---|
| Stoichiometric molar ratio $n_\text{KNO₃} : n_\text{sorb}$ | 26 : 5 |
| Stoichiometric O/F mass ratio | ≈ 2.22 |
| **KNO₃ mass fraction** | **≈ 68.9 %** |
| **Sorbitol mass fraction** | **≈ 31.1 %** |

These are the *stoichiometric* values — full combustion of sorbitol with
the oxygen supplied by KNO₃.

---
## 5. Why the operational mix is slightly fuel-rich (65% : 35%)

It has been established experimentally and confirmed by thermochemical codes
(e.g. NASA-CEA) that a slightly **fuel-rich** blend — typically 65% KNO₃ / 35%
sorbitol — outperforms the stoichiometric point in several respects:

In [8]:
operational = {
    "KNO₃"    : 0.65,
    "Sorbitol" : 0.35,
}

print("Operational KNSB mix (mass fractions):")
for species, frac in operational.items():
    print(f"  {species:<12}: {frac*100:.1f} %")

print(f"\nDeviation from stoichiometric KNO₃ fraction:  "
      f"{(operational['KNO₃'] - wt_KNO3)*100:+.2f} pp")

Operational KNSB mix (mass fractions):
  KNO₃        : 65.0 %
  Sorbitol    : 35.0 %

Deviation from stoichiometric KNO₃ fraction:  -9.27 pp


The excess sorbitol produces the following thermochemical advantages:

| Effect | Mechanism | Propulsion benefit |
|---|---|---|
| Sub-adiabatic flame temperature | Incomplete oxidation absorbs enthalpy | Reduced casing thermal load |
| More gas-phase products (CO, H₂) | Incomplete combustion raises moles of gas | Increased $I_{sp}$ |
| Lower $\text{K}_2\text{CO}_3$ yield | Oxygen limited below stoichiometric | Reduced nozzle slag / clogging |
| Stickier, more plastic slurry | Higher binder (sugar) loading | Better castability and grain integrity |

The trade-off is a marginal reduction in total energy release; the above benefits
are judged to outweigh this for hobby- and small experimental-rocket applications
(Nakka 2025).

---
## 6. Conclusion

Starting from the Mazza nullspace method (MATLAB, Chapter 2), we produced a
clean Python/SymPy drop-in replacement that:

- parses chemical formulas directly from symbolic expressions — no manual matrix entry;
- builds the signed balance matrix $\mathbf{A}$ algorithmically;
- computes the exact integer nullspace using `Matrix.nullspace()`;
- detects underdetermined systems (nullspace dim > 1) and reports them as warnings.

Applied to KNSB, the method confirmed that the 9-species product slate is
underdetermined and returned the **unique balanced equation** for the simplified
6-species system:

$$
\boxed{
26\,\text{KNO}_3 + 5\,\text{C}_6\text{H}_{14}\text{O}_6
\;\longrightarrow\;
17\,\text{CO}_2 + 35\,\text{H}_2\text{O} + 13\,\text{N}_2 + 13\,\text{K}_2\text{CO}_3
}
$$

The stoichiometric mass fraction of KNO₃ is **≈ 68.9%**, and
the operational blend (65% KNO₃ / 35% sorbitol) is chosen to sit
~4 percentage points fuel-rich of the stoichiometric point for
thermal, slag-minimisation, and castability reasons.

---
## References

1. D. Mazza and E. Canuto, *Fundamental Chemistry with MATLAB*, Elsevier, 2022.
2. R. Nakka, "KNSB Propellant," *Richard Nakka's Experimental Rocketry Web Site*, 2025. [Online]. Available: https://www.nakka-rocketry.net/sorb.html
3. J. Bonnie, J. Zehe, and S. Gordon, *NASA Glenn Coefficients for Calculating Thermodynamic Properties of Individual Species*, NASA/TP-2002-211556, Glenn Research Center, Cleveland, 2002.
4. W. Givens, "Parametric solution of linear homogeneous Diophantine equations," *Bull. Am. Math. Soc.*, vol. 53, no. 8, pp. 780–783, 1947.
